# SitePulse — Multi-factor Delay Risk Classification
### Phase 4: Supervised Project & Trade Delay Risk Prediction Engine

> **Simulation Disclaimer:**
> This notebook trains and evaluates delay risk classification models on synthetic construction fitout simulation datasets. The evaluation serves to establish algorithmic benchmarks and verify feature pipeline integrity. It does not represent real-world customer field deployments.

## 1. Setup & Ingestion of Day 25 Cutoff Feature Partitions

We load project-level feature vectors extracted at Day 25 cutoff. Preprocessing is strictly fit on training data only to avoid data leakage.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, brier_score_loss, accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

DATA_DIR = "../data/synthetic"
MODEL_DIR = "../model/risk_classifier"
RANDOM_SEED = 42

train_p = pd.read_csv(os.path.join(DATA_DIR, "train_features.csv"))
val_p = pd.read_csv(os.path.join(DATA_DIR, "val_features.csv"))
test_p = pd.read_csv(os.path.join(DATA_DIR, "test_features.csv"))

with open(os.path.join(MODEL_DIR, "feature_metadata.json")) as f:
    meta = json.load(f)
features = meta["project_features"]

X_train, y_train = train_p[features], train_p["target_is_delayed_over_3d"].values
X_val, y_val = val_p[features], val_p["target_is_delayed_over_3d"].values
X_test, y_test = test_p[features], test_p["target_is_delayed_over_3d"].values

print(f"Feature Count: {len(features)}")
print(f"Train Projects: {len(train_p)} ({y_train.sum()} delayed, {y_train.mean()*100:.1f}%)")
print(f"Val Projects  : {len(val_p)} ({y_val.sum()} delayed, {y_val.mean()*100:.1f}%)")
print(f"Test Projects : {len(test_p)} ({y_test.sum()} delayed, {y_test.mean()*100:.1f}%)")

## 2. Rule-Based Baseline Model

A simple rule-based heuristic classifying a project as delayed if `schedule_variance_pct < -5.0%` OR `velocity_health_ratio < 0.85`.

In [ ]:
rule_baseline_preds = ((val_p["schedule_variance_pct"] < -5.0) | (val_p["velocity_health_ratio"] < 0.85)).astype(int)

print("=== RULE-BASED BASELINE (VALIDATION SET) ===")
print(f"Precision: {precision_score(y_val, rule_baseline_preds) * 100:.2f}%")
print(f"Recall   : {recall_score(y_val, rule_baseline_preds) * 100:.2f}%")
print(f"F1-Score : {f1_score(y_val, rule_baseline_preds):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_val, rule_baseline_preds))

## 3. Supervised Model Architecture Comparison

We compare:
1. **Logistic Regression** (with median imputation and standard scaling, balanced class weights)
2. **Random Forest** (120 trees, balanced)
3. **HistGradientBoosting** (histogram gradient boosted decision trees)

In [ ]:
candidates = {
    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED))
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=120, max_depth=5, min_samples_split=4, class_weight="balanced", random_state=RANDOM_SEED))
    ]),
    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(max_iter=100, max_depth=4, learning_rate=0.08, random_state=RANDOM_SEED))
    ])
}

print("Evaluating candidates on Validation Set:")
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    val_probs = pipe.predict_proba(X_val)[:, 1]
    val_preds = pipe.predict(X_val)
    rec = recall_score(y_val, val_preds)
    prec = precision_score(y_val, val_preds, zero_division=0)
    f1 = f1_score(y_val, val_preds)
    roc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    print(f"{name:22s} | ROC-AUC: {roc:.4f} | PR-AUC: {pr_auc:.4f} | Recall: {rec*100:.1f}% | Precision: {prec*100:.1f}% | F1: {f1:.4f} | Brier: {brier:.4f}")

## 4. Threshold Calibration on Validation Set

In construction project management, missing a genuine delay carries massive liquidated damages and penalty costs. Therefore, we explicitly tune the decision threshold on validation data to achieve $\ge 90\%$ recall while maximizing precision.

In [ ]:
with open(os.path.join(MODEL_DIR, "threshold_config.json")) as f:
    thresh_cfg = json.load(f)
optimal_threshold = thresh_cfg["project_threshold"]
print(f"Loaded Tuned Threshold from Config: {optimal_threshold}")

## 5. Untouched Test Set Final Evaluation (Project Level)

We evaluate the serialized pipeline on the untouched test partition.

In [ ]:
model_pkg = joblib.load(os.path.join(MODEL_DIR, "risk_classifier.joblib"))
project_pipe = model_pkg["project_model"]

test_probs = project_pipe.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= optimal_threshold).astype(int)

cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

print("=== UNTOUCHED TEST SET PERFORMANCE (PROJECT LEVEL) ===")
print(f"Accuracy : {accuracy_score(y_test, test_preds) * 100:.2f}%")
print(f"Precision: {precision_score(y_test, test_preds) * 100:.2f}%")
print(f"Recall   : {recall_score(y_test, test_preds) * 100:.2f}%")
print(f"F1-Score : {f1_score(y_test, test_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, test_probs):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, test_probs):.4f}")
print(f"Brier    : {brier_score_loss(y_test, test_probs):.4f}")
print(f"Confusion Matrix:\nTP={tp}, FP={fp}, FN={fn}, TN={tn}")

## 6. Trade-Level Delay Risk Evaluation

In [ ]:
trade_pipe = model_pkg["trade_model"]
test_t = pd.read_csv(os.path.join(DATA_DIR, "test_trade_features.csv"))
trade_features = meta["trade_features"]

X_test_t = test_t[trade_features]
y_test_t = test_t["target_trade_is_delayed_over_3d"].values

trade_probs = trade_pipe.predict_proba(X_test_t)[:, 1]
trade_preds = (trade_probs >= thresh_cfg["trade_threshold"]).astype(int)

print("=== UNTOUCHED TEST SET PERFORMANCE (TRADE LEVEL) ===")
print(f"Accuracy : {accuracy_score(y_test_t, trade_preds) * 100:.2f}%")
print(f"Precision: {precision_score(y_test_t, trade_preds) * 100:.2f}%")
print(f"Recall   : {recall_score(y_test_t, trade_preds) * 100:.2f}%")
print(f"F1-Score : {f1_score(y_test_t, trade_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_t, trade_probs):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test_t, trade_probs):.4f}")
print(f"Brier    : {brier_score_loss(y_test_t, trade_probs):.4f}")

## 7. Model Explainability & Feature Importance

In [ ]:
print("Top Feature Weights (Logistic Regression Coefficients):")
for feat, imp in list(meta["feature_importances"].items())[:8]:
    print(f"  - {feat:30s}: {imp:+.4f}")

print("\nPermutation Importances (Validation Set):")
for feat, perm in list(meta["permutation_importance"].items())[:8]:
    print(f"  - {feat:30s}: {perm:+.4f}")

## 8. Human-Readable Prediction Explanation Demonstration

Example of the rule-augmented explainability engine generating operator-ready explanations from feature values available at prediction time.

In [ ]:
def explain_project_prediction(row):
    explanations = []
    if row["required_daily_velocity"] > 2.5:
        explanations.append(f"Required daily velocity is critically high ({row['required_daily_velocity']:.2f}%/day) to meet planned completion.")
    if row["schedule_variance_pct"] < -5.0:
        explanations.append(f"Overall project schedule variance is lagging significantly at {row['schedule_variance_pct']:.1f}%.")
    if row["workforce_fulfillment_ratio"] < 0.80:
        explanations.append(f"Workforce fulfillment is deficient at {row['workforce_fulfillment_ratio']*100:.1f}% of target staffing.")
    if row["velocity_health_ratio"] < 0.80:
        explanations.append(f"Recent 7-day progress velocity is failing to keep pace (health ratio: {row['velocity_health_ratio']:.2f}).")
    if row["active_delay_events_count"] > 0:
        explanations.append(f"Project is burdened by {int(row['active_delay_events_count'])} active delay events.")
    if not explanations:
        explanations.append("Project indicators are within normal operating margins; low delay risk.")
    return explanations

sample_delayed = test_p[test_p["target_is_delayed_over_3d"] == 1].iloc[0]
print(f"Sample Project: {sample_delayed['project_id']}")
sample_reasons = explain_project_prediction(sample_delayed)
for r in sample_reasons:
    print(f"  * {r}")